In [4]:
import json
import numpy as np
import pathlib
import os
import argparse
import re
from typing import Iterable, List, Dict, Any

from datasets import Dataset, load_dataset
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling, pipeline
from trl import SFTConfig, SFTTrainer


In [ ]:
base = "data/json_files_cleaned"
files = [
    f"{base}/ehac262.json",
    f"{base}/ehae176.json",
    f"{base}/ehae178.json",
    f"{base}/ehaf190.json",
    f"{base}/ehaf194.json",
]

"""ds_final = make_dataset_atomic(
    files,
)"""

raw_data = []

for f in files:
    ds = load_dataset("json", data_files=f, split="train")
    for row in ds:
        t_count = approx_tokens(row["text"])
        raw_data.append({
            "file": f.split('/')[-1],
            "tokens": t_count
        })

# Analiza pomoću Pandasa
df_raw = pd.DataFrame(raw_data)

print("=== ANALIZA SIROVIH PODATAKA (PRIJE OBRADE) ===")
print(f"Ukupno redova (stranica): {len(df_raw)}")
print(f"Ukupno tokena:           {df_raw['tokens'].sum()}")
print(f"Prosjek tokena po redu:  {df_raw['tokens'].mean():.2f}")
print(f"Minimalno tokena:        {df_raw['tokens'].min()}")
print(f"Maksimalno tokena:       {df_raw['tokens'].max()}")
print("-" * 47)

small_rows = len(df_raw[df_raw['tokens'] < 80])
print(f"Broj redova manjih od 80 tokena: {small_rows} ({small_rows/len(df_raw)*100:.1f}%)")


=== ANALIZA SIROVIH PODATAKA (PRIJE OBRADE) ===
Ukupno redova (stranica): 308
Ukupno tokena:           345056
Prosjek tokena po redu:  1120.31
Minimalno tokena:        165
Maksimalno tokena:       2250
-----------------------------------------------
Broj redova manjih od 80 tokena: 0 (0.0%)


Data loading for domain specific finetuning

In [3]:
ds = load_dataset("json", data_files="data/json_files_cleaned/ehaf194_gem.json", split="train")
print(ds)

Dataset({
    features: ['page', 'text'],
    num_rows: 73
})


In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE = "google/gemma-3-1b-it"  # stavi svoj točan naziv
base_tok = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto")

messages = [{"role": "user", "content": "Explain the guideline recommendation for mechanical heart valves in one paragraph."}]
inputs = base_tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
device = next(base_model.parameters()).device
inputs = inputs.to(device)

with torch.no_grad():
    out = base_model.generate(
        inputs, max_new_tokens=180, do_sample=False,
        repetition_penalty=1.15, no_repeat_ngram_size=3,
        eos_token_id=base_tok.eos_token_id, pad_token_id=base_tok.eos_token_id
    )

print(base_tok.decode(out[0], skip_special_tokens=True))



Some parameters are on the meta device because they were offloaded to the cpu and disk.


user
Explain the guideline recommendation for mechanical heart valves in one paragraph.
model
Mechanical heart valves are typically recommended based on a careful assessment of the patient’s individual circumstances, including the severity and type of heart failure they have, their overall health, and potential surgical risks. The goal is to select a valve that best suits their specific needs – whether it's a bioprosthetic valve (a natural tissue replacement) or an artificial valve (made from synthetic materials).  Guideline recommendations often prioritize valves with lower procedural complications like thrombosis and stenosis, considering factors such as age, comorbidities, and the patient's willingness to undergo surgery. Ultimately, the choice involves weighing benefits against risks and tailoring the valve selection to maximize long-term functionality and minimize impact on quality of life.ഞ്ഞുچندyjı‍️
ی‍️‍‍‍
I apologize for my previous response! It was inaccurate and didn'


Test run


In [4]:
"""# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)"""

'# Use a pipeline as a high-level helper\nfrom transformers import pipeline\n\npipe = pipeline("text-generation", model="google/gemma-3-1b-it")\nmessages = [\n    {"role": "user", "content": "Who are you?"},\n]\npipe(messages)'

In [5]:
#base model
model_name = "google/gemma-3-1b-it"


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [7]:
def tokenize(batch):
    return tokenizer(batch["text"], add_special_tokens=False, return_attention_mask=False)

#tokeniziranje dataseta   
tokenized_ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)


In [8]:
block_size = 256
#examples je jedan batch tokeniziranih podataka
def group_texts(examples):
    concatenated = []
    #input_ids je standardni naziv u Transformersima za:
    #tokenizirani tekst = lista brojeva (token ID-eva).
    for input_ids in examples["input_ids"]:
        concatenated.extend(input_ids)

    total_length = (len(concatenated) // block_size) * block_size
    #razdvajanje na blokove fiksne veličine
    concatenated = concatenated[:total_length]

    input_ids = [concatenated[i:i + block_size] for i in range(0, total_length, block_size)]
    return {"input_ids": input_ids, "labels": input_ids.copy()}

lm_ds = tokenized_ds.map(group_texts, batched=True)

In [9]:
print(lm_ds)
print(lm_ds[0].keys())                 # mora biti: input_ids, labels
print(len(lm_ds[0]["input_ids"]))      # mora biti 256
print(lm_ds[0]["input_ids"][:20])  

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 281
})
dict_keys(['input_ids', 'labels'])
256
[236865, 170431, 532, 82135, 236751, 108, 236778, 236796, 236787, 8512, 236772, 12108, 107, 236800, 236796, 236787, 14019, 236772, 12108, 107]


In [10]:
bnb_comfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_comfig)

In [12]:
#Lora config
lora_config = LoraConfig(
    r=8, #Veći r = više trenirajućih parametara = veća sposobnost učenja, ali više VRAM-a i sporije.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] ,#moduli koje ćemo prilagoditi LoR-om (quey, key, value, output projekcije u attention slojevima)
    lora_alpha=32, #veći alpha = veća težina LoRa prilagodbe u odnosu na osnovne težine.LoRA “jače utječe” (brže uči, ali može biti nestabilnije)
    lora_dropout=0.05, #regularizacija
    bias="none",
    task_type="CAUSAL_LM",
)

#W_new = W + (A @ B) * (alpha/r)
#W ostaje kvantiziran i “zamrznut”
#treniraš samo A i B (to su male matrice)Zato LoRA radi na slabijem hardveru.

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489


In [19]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)
args = TrainingArguments(
    output_dir="./gemma_lora_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    disable_tqdm=False,
    save_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds,
    data_collator=data_collator,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/lara/diplomski/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,4.001800
20,3.290300
30,3.271600


TrainOutput(global_step=36, training_loss=3.455517715877957, metrics={'train_runtime': 1689.8398, 'train_samples_per_second': 0.166, 'train_steps_per_second': 0.021, 'total_flos': 301866622844928.0, 'train_loss': 3.455517715877957, 'epoch': 1.0})

In [20]:
trainer.save_model("./gemma_lora_out")   # sprema adapter + trainer state
tokenizer.save_pretrained("./gemma_lora_out")


('./gemma_lora_out/tokenizer_config.json',
 './gemma_lora_out/special_tokens_map.json',
 './gemma_lora_out/chat_template.jinja',
 './gemma_lora_out/tokenizer.json')

In [28]:
import torch

model.eval()

messages = [{"role": "user", "content": "Explain the guideline recommendation for mechanical heart valves in one paragraph."}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)

# prebaci inpute na device gdje je model
device = next(model.parameters()).device
inputs = inputs.to(device)

with torch.no_grad():
    out = model.generate(
        inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


user
Explain the guideline recommendation for mechanical heart valves in one paragraph.
model
Okay, here's a breakdown of the recommended guidelines for mechanical valve management, explained in a single paragraph:

“The current standard practice for mechanical valves generally involves a staged approach guided by clinical factors and patient-specific risk assessments. The initial step is typically a **low-flow, low-pressure (LFLP) irrigation protocol** to minimize leaflet damage during implantation, followed by a gradual increase in flow rate over several weeks as tolerated.  A key component is **continuous monitoring of valve function**, including Doppler imaging and echocardiography, to assess valve performance and detect any signs of stenosis or regurgitation.  Furthermore, patients with significant mitral regurgitation are often prescribed a **mitral regurgitation (MR) valve repair**, which aims to improve


In [26]:
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

out = gen("Mechanical heart valves require lifelong", max_new_tokens=80, do_sample=True)
print(out[0]["generated_text"])

Device set to use cuda:0
/home/lara/diplomski/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/lara/diplomski/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Mechanical heart valves require lifelong intervention,XXANANANANANANANANANANANANARASASVSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSTSTSTSTDD
